# Anticipy Fara-7B QLoRA fine-tune

**Phase fara-5 of the FARA-7B integration build.**

This notebook fine-tunes a QLoRA adapter on top of `microsoft/Fara-7B` using the Anticipy synthetic trajectory dataset. The output adapter (~200 MB) is downloaded, merged into the base, and reconverted to MLX 4-bit as `fara-anticipy-v1` for local inference on the Mac.

## Where to run

Kaggle T4 free tier (16 GB VRAM). Add `microsoft/Fara-7B` and the Anticipy trajectory dataset as inputs.

## Wall-clock

Approximately 4 to 8 hours for 3 epochs over 400+ trajectories at sequence length 4096.

## Acceptance gate

Cold-test exact-match accuracy on action prediction (correct action type AND coordinate within 30 px) of at least 70 percent over the 50-trajectory held-out set. Below 60 percent means the trajectory data was insufficient or noisy; do NOT merge such an adapter into production.

In [ ]:
# Cell 1: dependencies (Kaggle preinstalls torch + transformers)
!pip install -q -U bitsandbytes peft accelerate datasets pillow trl wandb

In [ ]:
# Cell 2: paths + config
import os
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

BASE_MODEL = 'microsoft/Fara-7B'
TRAJ_INPUT_DIR = '/kaggle/input/anticipy-trajectories'  # the trajectory dataset uploaded to Kaggle
OUTPUT_DIR = '/kaggle/working/fara-anticipy-v1-adapter'
COLD_TEST_PATH = f'{TRAJ_INPUT_DIR}/cold_test.jsonl'
MAX_SEQ_LEN = 4096
IMAGE_LONG_EDGE = 1024  # downscale screenshots to 1024 long edge to fit T4 VRAM

import json
from pathlib import Path
trajectories = []
for jsonl_path in Path(TRAJ_INPUT_DIR).rglob('*.jsonl'):
    if jsonl_path.name == 'cold_test.jsonl':
        continue
    with jsonl_path.open() as fh:
        for line in fh:
            line = line.strip()
            if line:
                trajectories.append(json.loads(line))
print(f'training trajectories: {len(trajectories)}')
print(f'first scenario: {trajectories[0]["scenario"]}')

In [ ]:
# Cell 3: load Fara base model in 4-bit
import torch
from transformers import AutoProcessor, BitsAndBytesConfig, Qwen2_5_VLForConditionalGeneration

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
)
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map='auto',
    torch_dtype=torch.bfloat16,
)
processor = AutoProcessor.from_pretrained(BASE_MODEL)
print('model loaded')

In [ ]:
# Cell 4: attach LoRA adapters
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
model = prepare_model_for_kbit_training(model)
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# Cell 5: dataset adapter (one example per trajectory STEP, not per trajectory)
from PIL import Image
from torch.utils.data import Dataset

FARA_SYSTEM = (
    'You are a web automation agent that performs actions on websites '
    'to fulfill user requests by calling various tools.'
)

class TrajectorySteps(Dataset):
    def __init__(self, trajectories):
        self.flat = []
        for traj in trajectories:
            history = []
            for step in traj['steps']:
                self.flat.append((traj['goal'], history.copy(), step))
                history.append({
                    'thought': step.get('chain_of_thought', '')[:200],
                    'action': step['action'],
                })
    def __len__(self):
        return len(self.flat)
    def __getitem__(self, idx):
        goal, history, step = self.flat[idx]
        shot_path = f'{TRAJ_INPUT_DIR}/{step["screenshot"]}'
        img = Image.open(shot_path).convert('RGB')
        # Downscale long edge
        w, h = img.size
        if max(w, h) > IMAGE_LONG_EDGE:
            scale = IMAGE_LONG_EDGE / max(w, h)
            img = img.resize((int(w*scale), int(h*scale)), Image.LANCZOS)
        # Build messages
        history_text = ''
        for h_step in history[-5:]:
            history_text += f'Prior: {h_step["thought"]} Action: {json.dumps(h_step["action"])}\n'
        user_text = f'Task: {goal}\n{history_text}\nNext action?'
        target_text = (
            f'<think>{step.get("chain_of_thought", "")}</think>\n'
            f'{{"name": "computer_use", "arguments": {json.dumps(step["action"])}}}'
        )
        messages = [
            {'role': 'system', 'content': FARA_SYSTEM},
            {'role': 'user', 'content': [{'type':'image'}, {'type':'text','text':user_text}]},
            {'role': 'assistant', 'content': target_text},
        ]
        prompt = processor.apply_chat_template(messages, tokenize=False)
        encoded = processor(text=prompt, images=img, return_tensors='pt', padding=True, truncation=True, max_length=MAX_SEQ_LEN)
        encoded['labels'] = encoded['input_ids'].clone()
        return {k: v.squeeze(0) for k, v in encoded.items()}

ds = TrajectorySteps(trajectories)
print(f'training steps (flat): {len(ds)}')

In [ ]:
# Cell 6: train with HF Trainer
from transformers import TrainingArguments, Trainer
args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=64,
    learning_rate=2e-4,
    bf16=True,
    logging_steps=10,
    save_strategy='epoch',
    save_total_limit=2,
    warmup_ratio=0.03,
    report_to=['none'],  # set to ['wandb'] if wandb login
)
trainer = Trainer(model=model, args=args, train_dataset=ds, tokenizer=processor.tokenizer)
trainer.train()
model.save_pretrained(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)
print(f'adapter saved to {OUTPUT_DIR}')

In [ ]:
# Cell 7: evaluate cold test
if Path(COLD_TEST_PATH).exists():
    cold = []
    with open(COLD_TEST_PATH) as fh:
        for line in fh:
            line = line.strip()
            if line:
                cold.append(json.loads(line))
    print(f'cold-test trajectories: {len(cold)}')
    # Sample inference and compare predicted action vs ground truth
    correct = 0
    total = 0
    for traj in cold[:50]:
        for step in traj['steps']:
            total += 1
            shot_path = f'{TRAJ_INPUT_DIR}/{step["screenshot"]}'
            img = Image.open(shot_path).convert('RGB')
            messages = [
                {'role':'system','content':FARA_SYSTEM},
                {'role':'user','content':[{'type':'image'},{'type':'text','text':f'Task: {traj["goal"]}\nNext action?'}]},
            ]
            prompt = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            inputs = processor(text=prompt, images=img, return_tensors='pt').to(model.device)
            with torch.no_grad():
                out = model.generate(**inputs, max_new_tokens=128, temperature=0.0, do_sample=False)
            txt = processor.tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
            # Crude exact-match: action type + coordinate within 30 px
            try:
                import re
                m = re.search(r'"action"\s*:\s*"([^"]+)"', txt)
                pred_action = m.group(1) if m else ''
                gt_action = step['action'].get('action','')
                pred_match = pred_action == gt_action
                if 'coordinate' in step['action']:
                    cm = re.search(r'"coordinate"\s*:\s*\[\s*(\d+)\s*,\s*(\d+)\s*\]', txt)
                    if cm:
                        px, py = int(cm.group(1)), int(cm.group(2))
                        gx, gy = step['action']['coordinate']
                        coord_match = abs(px-gx) <= 30 and abs(py-gy) <= 30
                        pred_match = pred_match and coord_match
                if pred_match:
                    correct += 1
            except Exception:
                pass
    rate = correct / max(total, 1)
    print(f'cold-test exact match: {correct}/{total} = {100*rate:.1f}%')
    assert rate >= 0.70, f'cold-test rate {rate:.2f} below 0.70 threshold; do NOT merge adapter'
else:
    print(f'cold test not found at {COLD_TEST_PATH}; skipping')

In [ ]:
# Cell 8: merge adapter and download
merged = model.merge_and_unload()
MERGED_DIR = '/kaggle/working/fara-anticipy-v1-merged'
merged.save_pretrained(MERGED_DIR, safe_serialization=True)
processor.save_pretrained(MERGED_DIR)
print(f'merged model saved to {MERGED_DIR}')
# Tar for download
import shutil
shutil.make_archive('/kaggle/working/fara-anticipy-v1-adapter', 'zip', OUTPUT_DIR)
print('adapter zipped, ready to download from Kaggle output panel')